## Assignment 1: Data Wrangling, Visualization, and Web Presentation

### Data Extraction
**Method**: I will use Requests and BeautifulSoup to scrape the target table from wikipedia.

In [13]:
import requests
from bs4 import BeautifulSoup
import re
import sqlite3
import time
import json

WIKI_BASE = 'https://en.wikipedia.org'
URL = 'https://en.wikipedia.org/wiki/List_of_highest-grossing_films'
HEADERS = {'User-Agent': 'Mozilla/5.0'}

### Scraping pipeline:
1. First, we will fetch the page and find the target table
2. Then, we will iterate over the rows and extract the cells with data
3. We will populate: title, year, box_office, and link to the film's page
4. After that, we will scrape films' own pages to extract directors and countries

In [ ]:
def clean_money(raw: str) -> float | None:
    """Removes $ from value and casts it to float"""
    if not raw:
        return None

    match = re.search(r'\$[\d,]+', raw)
    if not match:
        return None

    num_str = match.group().replace('$', '').replace(',', '')

    try:
        return float(num_str)
    except ValueError:
        return None


def clean_year(raw: str) -> int | None:
    """Selects only the year and casts to int"""
    match = re.search(r'(\d{4})', raw)
    return int(match.group(1)) if match else None


def fetch_soup(url: str) -> BeautifulSoup:
    """Fetches the page"""
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, 'html.parser')


def clean_references(text: str) -> str:
    """Removes references like: [1], [ 1 ], [note 1]"""
    text = re.sub(r'\[\s*(?:note\s*|nb\s*)?[\w\d]+\s*\]', '', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()


def get_director_and_country(film_url: str) -> tuple[str, str]:
    """Visits individual page for each film to extract directors and country"""
    director, country = None, None

    try:
        soup = fetch_soup(WIKI_BASE + film_url)
        infobox = soup.find('table', class_='infobox')
        if not infobox:
            return director, country

        rows = infobox.find_all('tr')

        for row in rows:
            header = row.find("th")
            if not header:
                continue
            label = header.get_text(strip=True).lower()

            # Select director(s)
            if 'directed' in label or label == 'director':
                td = row.find("td")
                if td:
                    names = []

                    # Handles multiple directors for the case:
                    # <a><div class="plainlist"><ul>
                    #    <li>Anthony Russo</li>
                    #    <li>Joe Russo</li>
                    # </ul></div></a>
                    list_items = td.find_all('li')
                    if list_items:
                        names = [
                            li.get_text(strip=True) for li in list_items
                            if li.get_text(strip=True)
                        ]

                    # The second case - single director
                    # <a>James Cameron</a>
                    if not names:
                        anchors = td.find('a')
                        names = [anchors.get_text(strip=True)]

                    # Clean references and join
                    names = [clean_references(n) for n in names if n]
                    names = [n for n in names if n] # select not empty
                    director = ', '.join(names) if names else None

            # Select country
            if label in ('country', 'countries'):
                td = row.find('td')
                if td:
                    list_items = td.find_all('li')
                    if list_items: # Case with multiple countries
                        countries = [
                            li.get_text(strip=True) for li in list_items
                            if li.get_text(strip=True)
                        ]
                    else: # Case with single country
                        raw = td.get_text(strip=True)
                        countries = [raw] if raw else []

                    countries = [clean_references(c) for c in countries if c]
                    countries = [c for c in countries if c]
                    country = ", ".join(countries) if countries else None

    except Exception as e:
        print(f'Could not fetch details from {film_url}: {e}')

    return director, country

### Fetching whole page and finding the target table

In [19]:
soup = fetch_soup(URL)

# Select all the tables
tables = soup.find_all('table', class_='wikitable')

# Find the only one with the ranks
main_table = None
for tbl in tables:
    first_row_headers = [th.get_text(strip=True) for th in tbl.find_all('th')]
    if 'Rank' in first_row_headers and 'Title' in first_row_headers:
        main_table = tbl
        break

if main_table is None:
    main_table = tables[0]
    print('Could not locate the expected table; using the first wikitable.')

print('Main table located.')

Main table located.


### Parsing the table and data cleaning

In [20]:
films = []

rows = main_table.find_all('tr')[1:] # Select all the rows

for row in rows:
    cells = row.find_all(['th', 'td']) # Extract cells

    if len(cells) < 4:
        continue

    title_cell = cells[2]
    title = title_cell.get_text(strip=True) # Select title

    # Select film link to get the directors and country
    title_tag = title_cell.find('a')
    film_link = title_tag['href'] if title_tag and title_tag.has_attr('href') else None

    # Select gross
    gross_raw = cells[3].get_text(strip=True)
    box_office = clean_money(gross_raw)

    # Select year
    year_raw = cells[4].get_text(strip=True)
    year = clean_year(year_raw)

    films.append({
        'title': title,
        'year': year,
        'box_office': box_office,
        'link': film_link,
        'director': None,
        'country': None,
    })

print(f'Extracted {len(films)} films from the main table.')
print(films)

Extracted 50 films from the main table.
[{'title': 'Avatar', 'year': 2009, 'box_office': 2923710708.0, 'link': '/wiki/Avatar_(2009_film)', 'director': None, 'country': None}, {'title': 'Avengers: Endgame', 'year': 2019, 'box_office': 2797501328.0, 'link': '/wiki/Avengers:_Endgame', 'director': None, 'country': None}, {'title': 'Avatar: The Way of Water', 'year': 2022, 'box_office': 2334484620.0, 'link': '/wiki/Avatar:_The_Way_of_Water', 'director': None, 'country': None}, {'title': 'Titanic', 'year': 1997, 'box_office': 2257906828.0, 'link': '/wiki/Titanic_(1997_film)', 'director': None, 'country': None}, {'title': 'Ne Zha 2', 'year': 2025, 'box_office': 2215690000.0, 'link': '/wiki/Ne_Zha_2', 'director': None, 'country': None}, {'title': 'Star Wars: The Force Awakens', 'year': 2015, 'box_office': 2068223624.0, 'link': '/wiki/Star_Wars:_The_Force_Awakens', 'director': None, 'country': None}, {'title': 'Avengers: Infinity War', 'year': 2018, 'box_office': 2048359754.0, 'link': '/wiki/Av

### Fetching directors and countries

In [27]:
for i, film in enumerate(films):
    if film['link']:
        director, country = get_director_and_country(film['link'])
        film['director'] = director
        film['country']  = country
        print(f"[{str(i+1):2s}/{len(films)}] {film['title']:30s} -> dir: {director:30s} | c: {country}")
        time.sleep(0.5) # wait between the requests

print('Director/country enrichment complete.')

[1 /50] Avatar                         -> dir: James Cameron                  | c: United States, United Kingdom
[2 /50] Avengers: Endgame              -> dir: Anthony Russo, Joe Russo       | c: United States
[3 /50] Avatar: The Way of Water       -> dir: James Cameron                  | c: United States
[4 /50] Titanic                        -> dir: James Cameron                  | c: United States
[5 /50] Ne Zha 2                       -> dir: Jiaozi                         | c: China
[6 /50] Star Wars: The Force Awakens   -> dir: J. J. Abrams                   | c: United States
[7 /50] Avengers: Infinity War         -> dir: Anthony Russo, Joe Russo       | c: United States
[8 /50] Spider-Man: No Way Home        -> dir: Jon Watts                      | c: United States
[9 /50] Zootopia 2†                    -> dir: Jared Bush                     | c: United States
[10/50] Inside Out 2                   -> dir: Kelsey Mann                    | c: United States
[11/50] Jurassic World

## Database
I will use SQLite for simplicity

In [ ]:
DB_NAME = 'highest_grossing_films.db'

# Connect to db
conn = sqlite3.connect(DB_NAME)
cur  = conn.cursor()

# Create the table
cur.execute('DROP TABLE IF EXISTS films;')
cur.execute("""
    CREATE TABLE films (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        title        TEXT    NOT NULL,
        release_year INTEGER,
        director     TEXT,
        box_office   REAL,
        country      TEXT
    );
""")

# Insert values
insert_sql = """
    INSERT INTO films (title, release_year, director, box_office, country)
    VALUES (?, ?, ?, ?, ?);
"""
for f in films:
    cur.execute(insert_sql, (
        f['title'],
        f['year'],
        f['director'],
        f['box_office'],
        f['country'],
    ))

# Save changes
conn.commit()

print(f"Inserted {len(films)} rows into '{DB_NAME}'.")

Inserted 50 rows into 'highest_grossing_films.db'.


### Place the data to `films.json` to use it in GitHub pages

In [ ]:
import os


JSON_NAME = 'data/films.json'

conn = sqlite3.connect(DB_NAME)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("""
    SELECT id,
           title,
           release_year,
           director,
           box_office,
           country
    FROM   films
    ORDER  BY box_office DESC;
""")

films = [dict(row) for row in cur.fetchall()]
conn.close()

# Check if the folder exists
os.makedirs('data', exist_ok=True)

with open(JSON_NAME, 'w', encoding='utf-8') as f:
    json.dump(films, f, indent=2, ensure_ascii=False)

print(f'Exported {len(films)} films to {JSON_NAME}')

Exported 50 films to data/films.json
